In [5]:
import json

# 转换数据函数
def convert_to_instruction_format(data):
    # 构造 entities
    entities = [{"text": entity["text"], "label": entity["label"]} for entity in data["entityMentions"]]

    # 构造 relations
    relations = [
        {"head": relation["em1Text"], "tail": relation["em2Text"], "label": relation["label"]}
        for relation in data["relationMentions"]
    ]

    # 构造输出 JSON
    result = {
        "instruction": (
            "你是一个文本关系实体识别领域的专家，你需要从给定的句子中提取: 人名实体(Nh); 地名实体(Ns); 时间实体(NT); 毒品类型实体(NDR); 毒品重量实体(NW); "
            "关系分别为: 贩卖给人(sell_drug_to); 贩卖毒品(traffic_in); 持有(possess); 非法容留(provide_shelter_for). 以 json 格式输出, "
            "如 {\"entities\":[{\"text\": \"林某某\", \"label\": \"Nh\"}],\"relations\":[{\"head\": \"林某某\", \"tail\": \"海洛因\", \"label\": \"traffic_in\"}]} "
            "注意: 1. 输出的每一行都必须是正确的 json 字符串. 2. 找不到任何实体或关系时, 输出{\"entities\":[],\"relations\":[]}."
        ),
        "input": f"文本: {data['sentText']}",
        "output": json.dumps({"entities": entities, "relations": relations}, ensure_ascii=False)
    }

    return result

# 处理逐行 JSON 文件
def process_line_by_line_json(input_file, output_file):
    with open(input_file, 'r', encoding='utf-8') as infile, open(output_file, 'w', encoding='utf-8') as outfile:
        for line in infile:
            # 跳过空行
            if not line.strip():
                continue
            # 读取每行 JSON
            data = json.loads(line.strip())
            # 转换为指令微调格式
            converted_data = convert_to_instruction_format(data)
            # 写入到输出文件中，每行一个 JSON 对象
            outfile.write(json.dumps(converted_data, ensure_ascii=False) + '\n')

    print(f"转换完成，结果已保存到 {output_file}")

# 使用示例
input_file = "/Users/yuu/Downloads/毕业设计/CAIL2022-main/xxcq/train.json"  # 输入文件路径
output_file = "/Users/yuu/Downloads/毕业设计/final_train.jsonl"  # 输出文件路径
process_line_by_line_json(input_file, output_file)  

转换完成，结果已保存到 /Users/yuu/Downloads/毕业设计/final_train.jsonl


In [7]:
import pandas as pd 
total_df = pd.read_json('/Users/yuu/Downloads/毕业设计/final_train.jsonl', lines=True)


Qwen -- split

In [7]:
import json

# Input file
input_file = '/Users/yuu/Downloads/毕业设计/CAIL2022-main/xxcq/test.json'
ner_output_file = '/Users/yuu/Downloads/毕业设计/CAIL2022-main/xxcq/ner_test.jsonl'
re_output_file = '/Users/yuu/Downloads/毕业设计/CAIL2022-main/xxcq/re_test.jsonl'

# Prepare outputs
ner_instructions = []
re_instructions = []

# Process data line by line
with open(input_file, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line.strip())
        sent_text = item["sentText"]
        entities = item["entityMentions"]
        relations = item["relationMentions"]

        # Create NER task instruction
        ner_instruction = {
            "instruction": "你是一个命名实体识别领域的专家，你需要从给定的句子中提取以下实体: 人名实体(Nh); 地名实体(Ns); 时间实体(NT); 毒品类型实体(NDR); 毒品重量实体(NW); 以 json 格式输出, "
            "如 {\"text\": \"林某某\", \"label\": \"Nh\"} "
            "注意: 1. 输出的每一行都必须是正确的 json 字符串. 2. 找不到任何实体或关系时, 输出\"找不到任何实体\"",
            "input": sent_text,
            "output": [
                {"text": ent["text"], "label": ent["label"]}
                for ent in entities
            ],
        }
        ner_instructions.append(ner_instruction)

        # Create RE task instruction
        re_instruction = {
            "instruction": "你是一个关系抽取领域的专家，你需要从给定的句子中提取以下关系:"
            " 贩卖给人(sell_drug_to); 贩卖毒品(traffic_in); 持有(possess); 非法容留(provide_shelter_for). 以 json 格式输出, "
            "如{\"em1Text\": \"林某某\", \"em2Text\": \"海洛因\", \"label\": \"traffic_in\"} "
            "注意: 1. 输出的每一行都必须是正确的 json 字符串. 2. 找不到任何关系时, 输出\"找不到任何关系\"",
            "input": sent_text,
            "output": [
                {"em1Text": rel["em1Text"], "em2Text": rel["em2Text"], "label": rel["label"]}
        for rel in relations if rel["label"] != "NA"
            ],
        }
        re_instructions.append(re_instruction)

# Write NER JSONL file
with open(ner_output_file, "w", encoding="utf-8") as f:
    for instruction in ner_instructions:
        f.write(json.dumps(instruction, ensure_ascii=False) + "\n")

# Write RE JSONL file
with open(re_output_file, "w", encoding="utf-8") as f:
    for instruction in re_instructions:
        f.write(json.dumps(instruction, ensure_ascii=False) + "\n")

print(f"NER training data written to {ner_output_file}")
print(f"RE training data written to {re_output_file}")

NER training data written to /Users/yuu/Downloads/毕业设计/CAIL2022-main/xxcq/ner_test.jsonl
RE training data written to /Users/yuu/Downloads/毕业设计/CAIL2022-main/xxcq/re_test.jsonl


In [ ]:
import pandas as pd
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score
import json

def extract_entities(entity_list):
    """
    Convert the string representation of entity lists to Python objects and extract entities as tuples.
    """
    try:
        if isinstance(entity_list, str):
            entities = json.loads(entity_list.replace("'", '"'))  # 转换为标准 JSON 格式
        else:
            entities = entity_list
        return [(entity['text'], entity['label']) for entity in entities]
    except Exception as e:
        print(f"Error parsing entity_list: {entity_list}. Exception: {e}")
        return []  # Return empty list for problematic rows

# Load the data
file_path = '/Users/yuu/Downloads/ner_predictions-2.csv'  # Replace with your file path
ner_data = pd.read_csv(file_path)

ner_data['gold_entities_extracted'] = ner_data['gold_entities'].apply(extract_entities)
ner_data['predicted_entities_extracted'] = ner_data['predicted_entities'].apply(extract_entities)

# Flatten lists for comparison
gold_flat = []
pred_flat = []

for index, row in ner_data.iterrows():
    gold_entities = row['gold_entities_extracted']
    pred_entities = row['predicted_entities_extracted']
    # Only include rows where the number of entities matches
    if len(gold_entities) == len(pred_entities):
        gold_flat.extend(gold_entities)
        pred_flat.extend(pred_entities)
    else:
        print(f"Skipping row {index} due to mismatch: gold={len(gold_entities)}, pred={len(pred_entities)}")

# Separate texts and labels for classification report
gold_labels = [label for _, label in gold_flat]
pred_labels = [label for _, label in pred_flat]

# Check consistency
if len(gold_labels) != len(pred_labels):
    raise ValueError(f"Inconsistent lengths: gold_labels={len(gold_labels)}, pred_labels={len(pred_labels)}")

# Generate classification report
classification_report_result = classification_report(gold_labels, pred_labels, zero_division=0, output_dict=False)
precision_score_ = precision_score(gold_labels, pred_labels, average='macro', zero_division=0)
recall_ = recall_score(gold_labels, pred_labels, average='macro', zero_division=0)
f1_ = f1_score(gold_labels, pred_labels, average='macro', zero_division=0)

# Output results
print(f"Precision: {precision_score_}")
print(f"Recall: {recall_}")
print(f"F1: {f1_}")
print("Classification Report:")
print(classification_report_result)


Error parsing entity_list: [{'text': '2016年11月1日15时30分', 'label': 'NT'}, {'text': '肖某某', 'label': 'Nh'}, {'text': '贵阳市南明区玉田巷', 'label': 'Ns'}, {'text': '海洛因', 'label': 'NDR'}, {'text': '徐某"4', 'label': 'Nh'}, {'text': '肖某某', 'label': 'Nh'}, {'text': '徐某"4', 'label': 'Nh'}, {'text': '肖某某', 'label': 'Nh'}, {'text': '海洛因', 'label': 'NDR'}, {'text': '0.10克', 'label': 'NW'}, {'text': '肖某某', 'label': 'Nh'}, {'text': '海洛因', 'label': 'NDR'}, {'text': '0.46克', 'label': 'NW'}, {'text': '2016年11月7日', 'label': 'NT'}, {'text': '肖某某', 'label': 'Nh'}, {'text': '徐某', 'label': 'Nh'}, {'text': '海洛因', 'label': 'NDR'}, {'text': '0.56克', 'label': 'NW'}, {'text': '海洛因', 'label': 'NDR'}]. Exception: Expecting ',' delimiter: line 1 column 163 (char 162)
Error parsing entity_list: [{'text': '2016年7月19日16时', 'label': 'NT'}, {'text': '曹某', 'label': 'Nh'}, {'text': '凯里市环城西路114号省林汽宿舍11号家中的书房', 'label': 'Ns'}, {'text': '罗某', 'label': 'Nh'}, {'text': '张某"4', 'label': 'Nh'}, {'text': '谢某', 'label': 'Nh'}, {'text': '滕

In [5]:
import pandas as pd
import ast

def calculate_accuracy(df):
    correct_predictions = 0
    total_gold_entities = 0

    for _, row in df.iterrows():
        # Parse the gold and predicted entities from string to list of dictionaries
        gold_entities = ast.literal_eval(row['gold_entities'])
        predicted_entities = ast.literal_eval(row['predicted_entities'])

        # Convert to sets of (label, text) for comparison
        gold_set = set((entity['label'], entity['text']) for entity in gold_entities)
        predicted_set = set((entity['label'], entity['text']) for entity in predicted_entities)

        # Calculate correct predictions and total gold entities
        correct_predictions += len(gold_set & predicted_set)
        total_gold_entities += len(gold_set)

    # Avoid division by zero
    accuracy = correct_predictions / total_gold_entities if total_gold_entities > 0 else 0
    return accuracy

# Load the dataset
file_path = '/Users/yuu/Downloads/ner_predictions-2.csv' 
data = pd.read_csv(file_path)

# Calculate and display the accuracy
accuracy = calculate_accuracy(data)
print(f"Entity Recognition Accuracy: {accuracy:.2%}")

Entity Recognition Accuracy: 83.69%


In [8]:
import pandas as pd
import ast
from sklearn.metrics import classification_report

def calculate_metrics(df):
    y_true = []
    y_pred = []

    for _, row in df.iterrows():
        # Parse the gold and predicted entities from string to list of dictionaries
        gold_entities = ast.literal_eval(row['gold_entities'])
        predicted_entities = ast.literal_eval(row['predicted_entities'])

        # Convert to sets of (label, text) for comparison
        gold_set = set((entity['label'], entity['text']) for entity in gold_entities)
        predicted_set = set((entity['label'], entity['text']) for entity in predicted_entities)

        # Append labels for classification report
        for entity in gold_set:
            y_true.append(entity[0])  # Append label of gold entities
            y_pred.append(entity[0] if entity in predicted_set else "O")  # Match or "O" for missing predictions

        for entity in predicted_set - gold_set:
            y_true.append("O")  # Not in gold entities
            y_pred.append(entity[0])  # Predicted entity label

    # Generate classification report
    report = classification_report(y_true, y_pred, labels=list(set(y_true) - {"O"}), zero_division=0, output_dict=True)

    # Extract overall metrics
    precision = report['macro avg']['precision']
    recall = report['macro avg']['recall']
    f1_score = report['macro avg']['f1-score']

    return precision, recall, f1_score, report

# Load the dataset
file_path = '/Users/yuu/Downloads/ner_predictions-2.csv' 
data = pd.read_csv(file_path)

# Calculate metrics
precision, recall, f1_score, report = calculate_metrics(data)

# Print metrics
print(f"Precision: {precision:.2%}")
print(f"Recall: {recall:.2%}")
print(f"F1 Score: {f1_score:.2%}")
print("\nClassification Report:")
print(pd.DataFrame(report).transpose())

Precision: 82.17%
Recall: 81.70%
F1 Score: 81.93%

Classification Report:
              precision    recall  f1-score  support
NDR            0.960152  0.952919  0.956522    531.0
Nh             0.958432  0.940559  0.949412    858.0
NT             0.807910  0.803371  0.805634    534.0
Ns             0.494331  0.486607  0.490439    448.0
NW             0.887435  0.901596  0.894459    376.0
micro avg      0.844289  0.836913  0.840585   2747.0
macro avg      0.821652  0.817010  0.819293   2747.0
weighted avg   0.844097  0.836913  0.840462   2747.0


In [4]:
import pandas as pd
from sklearn.metrics import classification_report, precision_recall_fscore_support

# 读取数据
file_path = '/Users/yuu/Downloads/ner_predictions.csv'  # 替换为您的文件路径
data = pd.read_csv(file_path)

# 确保数据包含 'gold_entities' 和 'predicted_entities' 列
if 'gold_entities' in data.columns and 'predicted_entities' in data.columns:
    gold_entities = data['gold_entities']
    predicted_entities = data['predicted_entities']
    
    # 计算总体 Precision、Recall 和 F1
    precision, recall, f1, _ = precision_recall_fscore_support(
        gold_entities, predicted_entities, average='weighted', zero_division=0
    )
    
    # 输出结果
    print(f"Overall Precision: {precision:.4f}")
    print(f"Overall Recall: {recall:.4f}")
    print(f"Overall F1-score: {f1:.4f}")
else:
    print("The file does not contain the required columns 'gold_entities' and 'predicted_entities'.")

Overall Precision: 0.6275
Overall Recall: 0.6289
Overall F1-score: 0.6280
